# PIMC - Proton in a double well potential

In [ ]:
import numpy as np
import scipy.constants as const

# Constants
P = 5 # Number of bids 
N = 10e4 # Number of iterations
a = 0.04e-9 # Potential lenght parameter
C = 0.3 # Unit reduction factor
m = const.m_e # electron mass
V0 = const.hbar**2 / (2 * m * a**2 * C) # Potential energy parameter
T = 0.4 * V0 * const.k # Simulation temperature
dx = 0.1 * a # Spatial step size (to be fine-tuned)
omega_P = np.sqrt(P) * const.k * T / const.hbar # Quantum spring frequency

# Initialization
M_conf = np.ones((1, 5)) # Configuration matrix for bids
success_list = np.array([]) # List to store success rates

# Functions
def random_displacement(dx):
    """
    Generate a random displacement following dx*(eta-0.5), where eta is a uniform random number in [0, 1].
    """
    eta = np.random.uniform(0, 1)
    return dx * (2 * eta - 1)

def V(x):
    """
    Potential energy function.
    """
    return V0 * ((x / a)**2 - 1)**2

def Delta_U(state, bid_index, displacement):
    """
    Calculate the change in potential energy for a given bid index and displacement.
    """
    # Compute the diff in potential energy
    V_diff = (1/P) * (V(state[bid_index] + displacement) - V(state[bid_index]))

    # Compute the diff in spring energy
    xj_old = state[bid_index]
    xj_new = xj_old + displacement
    xjp = state[(bid_index + 1) % P]
    xjm = state[(bid_index - 1) % P]
    spring_diff = 0.5 * m * omega_P**2 * ((xjp - xj_new)**2 + (xj_new - xjm)**2 - (xjp - xj_old)**2 - (xj_old - xjm)**2)
    return V_diff + spring_diff

def metropolis_acceptance(delta_U):
    """
    Metropolis acceptance criterion.
    """
    if delta_U < 0:
        return True
    else:
        acceptance_prob = np.exp(-delta_U / (const.k * T))
        return np.random.uniform(0, 1) < acceptance_prob

def algo_step(M_conf, success_list):
    """
    Perform a single step of the Metropolis algorithm.
    """
    # Randomly select a bid index
    bid_index = np.random.randint(0, P)

    # Generate a random displacement
    displacement = random_displacement(dx)

    # Calculate the change in potential energy
    delta_U = Delta_U(M_conf[-1], bid_index, displacement)

    # Decide whether to accept the move
    M_conf = np.vstack([M_conf, M_conf[-1].copy()])
    if metropolis_acceptance(delta_U):
        M_conf[-1][bid_index] += displacement
        success_list = np.append(success_list, 1)  # Move accepted
        return True  # Move accepted
    else:
        success_list = np.append(success_list, 0)  # Move rejected
        return False  # Move rejected

def run_simulation(N):
    """
    Run the Metropolis algorithm for N iterations.
    """
    for _ in range(int(N)):
        algo_step(M_conf, success_list)

